In [0]:
import os
account_key = os.environ.get("AZURE_STORAGE_KEY")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7802622528979289>, line 1
----> 1 spark.conf.set(
      2     "fs.azure.account.key.hisadls.dfs.core.windows.net",
      3     "HUEWkIOPAky364XxhPkoxuRAMFzjSXrPU6XvL6TXco0Q7dAu3bSWjocqhjY3Tj8/YIXQ+DGslARo+AStE7+tIA=="
      4 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warnings:
     48     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2246, in SparkConnectClient.config(self, operation)
   2244     raise SparkConnectException("Invalid state during retry exceptio

In [0]:
from pyspark.sql.functions import *

table_name = "acx_testdetails"

raw_path = f"abfss://cdcbronze@hisadls.dfs.core.windows.net/{table_name}/full_load"

df_raw = spark.read.format("parquet").load(raw_path)
df_raw.limit(3).display()

COSTCENTER,TESTINGCENTER,BOOKINGDATE,APPOINTMENTDATE,SALESID,PATH_NONPATH,DEPARTMENT,MODELGROUPID,ITEMID,TESTSTATUS,CUSTOMER_TYPE
ASANSOL,DRL,2026-01-01,null,PR1423433,None,PACK,PACK,PACK0005,VerifiedByDoctor,DIRECT CLIENT
ASANSOL,LAB,2026-01-01,null,PR1423930,None,PACK,PACK,PACK0193,VerifiedByDoctor,GENERAL
ASANSOL,DRL,2026-01-01,null,PR1423433,None,PACK,PACK,PACK1081,VerifiedByDoctor,DIRECT CLIENT


In [0]:
cdc_path = f"abfss://cdcbronze@hisadls.dfs.core.windows.net/{table_name}/cdc"

cdc_df = spark.read.format("parquet").load(cdc_path)
cdc_df.limit(3).display()

__$start_lsn,__$seqval,__$operation,__$update_mask,COSTCENTER,TESTINGCENTER,BOOKINGDATE,APPOINTMENTDATE,SALESID,PATH_NONPATH,DEPARTMENT,MODELGROUPID,ITEMID,TESTSTATUS,CUSTOMER_TYPE
AAAAPQAAHsAAGA==,AAAAPQAAHsAABQ==,2,B/8=,BEHALA,LAB,2026-01-02,null,PR1424829,None,PACK,PACK,PACK0002,VerifiedByDoctor,CORPORATE
AAAAPQAAHsAAGA==,AAAAPQAAHsAABg==,2,B/8=,BEHALA,LAB,2026-01-02,null,PR1426776,None,PACK,PACK,PACK0002,VerifiedByDoctor,CORPORATE
AAAAPQAAHsAAGA==,AAAAPQAAHsAABw==,2,B/8=,BEHALA,LAB,2026-01-02,null,PR1424930,None,PACK,PACK,PACK0005,VerifiedByDoctor,CORPORATE


%md
##### - Never merge raw table directly.
##### - Always deduplicate first.

In [0]:
# Keep Only Required Operations

r_cdc = cdc_df.filter(col("__$operation").isin(1,2,4))

# We must keep latest to deduplicate

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy(concat(col("SALESID"), lit("-"), col("ITEMID"))).orderBy(desc("__$start_lsn"), desc("__$seqval"))

df_latest = r_cdc.withColumn("rn", row_number().over(window_spec))\
    .filter(col("rn") == 1)\
        .drop("rn")

#### limit(0) This creates: Empty Delta table, Only schema, No data

In [0]:
silver_path = f"abfss://cdcsilver@hisadls.dfs.core.windows.net/{table_name}"

from delta.tables import DeltaTable

# Ensure table exists

if not DeltaTable.isDeltaTable(spark, silver_path):
    df_raw.write.format("delta").save(silver_path)

# Load Delta table

silver_table = DeltaTable.forPath(spark, silver_path)

silver_table.alias("T")\
    .merge(
        df_latest.alias("S"),
        "T.SALESID = S.SALESID AND T.ITEMID = S.ITEMID"
    ).whenMatchedDelete(
        condition= "S.`__$operation` = 1"
    ).whenMatchedUpdate(
        condition= "S.`__$operation` = 4",
        set= {col: f"S.{col}" for col in df_latest.columns if not col.startswith("__$")}
    ).whenNotMatchedInsert(
        condition= "S.`__$operation` = 2",
        values= {col: f"S.{col}" for col in df_latest.columns if not col.startswith("__$")}
    ).execute()


In [0]:
spark.read.format("delta").load(silver_path).count()

In [0]:
from pyspark.sql.functions import max

last_lsn = df_latest.select(max("__$start_lsn")).collect()[0][0]

dbutils.notebook.exit(last_lsn.hex())